In [1]:
# Mounting your Google Drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [20]:
# Step (2): Reading the dataset and turn it into a Python dataframe

In [3]:
import pandas as pd
df = pd.read_csv('/content/drive/My Drive/heart.csv')

In [7]:
# Step (3): Exploring the dataset

In [5]:
# General information about the dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1025 entries, 0 to 1024
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1025 non-null   int64  
 1   sex       1025 non-null   int64  
 2   cp        1025 non-null   int64  
 3   trestbps  1025 non-null   int64  
 4   chol      1025 non-null   int64  
 5   fbs       1025 non-null   int64  
 6   restecg   1025 non-null   int64  
 7   thalach   1025 non-null   int64  
 8   exang     1025 non-null   int64  
 9   oldpeak   1025 non-null   float64
 10  slope     1025 non-null   int64  
 11  ca        1025 non-null   int64  
 12  thal      1025 non-null   int64  
 13  target    1025 non-null   int64  
dtypes: float64(1), int64(13)
memory usage: 112.2 KB


In [7]:
# Shape of the dataframe
print ("Number of Data Records:", len(df))
print ("Dataset Shape:", df.shape)

Number of Data Records: 1025
Dataset Shape: (1025, 14)


In [10]:
# Showing some data records
df.head(121)

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116,63,1,0,130,254,0,0,147,0,1.4,1,1,3,0
117,43,1,0,120,177,0,0,120,1,2.5,1,0,3,0
118,29,1,1,130,204,0,0,202,0,0.0,2,0,2,1
119,42,1,1,120,295,0,1,162,0,0.0,2,0,2,1


In [11]:
# Accessing a certain row/column
#dataset.loc[2].at["Dust Allergy"]
df.loc[0].iat[0]

np.float64(52.0)

In [12]:
# Accessing a certain row
df.iloc[121]

,121
age,44.0
sex,1.0
cp,0.0
trestbps,120.0
chol,169.0
fbs,0.0
restecg,1.0
thalach,144.0
exang,1.0
oldpeak,2.8


In [13]:
# Step (4): Prepare the dataset for supervised learning
## 1. Separate features (X) and target variable (y)
X = df.drop('target', axis=1)  # Feature variables (exclude the 'target' column)
y = df['target']               # Target variable

In [14]:
from sklearn import preprocessing

## 2. Data normalization (Min-Max scaling)
min_max_scaler = preprocessing.MinMaxScaler()
X_scaled = min_max_scaler.fit_transform(X)  # Normalize all features
X = pd.DataFrame(X_scaled, columns=X.columns)

In [15]:
from sklearn.model_selection import KFold

## 3. Define K-fold cross-validation (K=5)
cv_data = KFold(n_splits=5, random_state=1, shuffle=True)

In [16]:
from sklearn.model_selection import cross_validate
from sklearn.naive_bayes import GaussianNB

# Step (5): Build and evaluate Decision Tree and Naive Bayes models
## Naive Bayes model (GaussianNB)
print("\n=== Naive Bayes Model Evaluation ===")
results_NB = cross_validate(
    GaussianNB(), X, y, cv=cv_data,
    scoring=['accuracy', 'precision_macro', 'recall_macro'],  # Evaluation metrics: Accuracy, Precision, Recall
    return_train_score=True
)


=== Naive Bayes Model Evaluation ===


In [18]:
import numpy as np
# Print training/test metrics for Naive Bayes
print(f"Training Accuracy: {np.around(results_NB['train_accuracy'], 3)}, Mean: {results_NB['train_accuracy'].mean():.2f}")
print(f"Test Accuracy: {np.around(results_NB['test_accuracy'], 3)}, Mean: {results_NB['test_accuracy'].mean():.2f}")
print(f"Test Precision (Macro Average): {results_NB['test_precision_macro'].mean():.2f}")
print(f"Test Recall (Macro Average): {results_NB['test_recall_macro'].mean():.2f}")

Training Accuracy: [0.843 0.828 0.821 0.818 0.826], Mean: 0.83
Test Accuracy: [0.78  0.805 0.839 0.844 0.834], Mean: 0.82
Test Precision (Macro Average): 0.82
Test Recall (Macro Average): 0.82


In [19]:
from sklearn.tree import DecisionTreeClassifier

## Decision Tree model
print("\n=== Decision Tree Model Evaluation ===")
results_DT = cross_validate(
    DecisionTreeClassifier(random_state=1), X, y, cv=cv_data,
    scoring=['accuracy', 'precision_macro', 'recall_macro'],
    return_train_score=True
)


=== Decision Tree Model Evaluation ===


In [20]:
# Print training/test metrics for Decision Tree
print(f"Training Accuracy: {np.around(results_DT['train_accuracy'], 3)}, Mean: {results_DT['train_accuracy'].mean():.2f}")
print(f"Test Accuracy: {np.around(results_DT['test_accuracy'], 3)}, Mean: {results_DT['test_accuracy'].mean():.2f}")
print(f"Test Precision (Macro Average): {results_DT['test_precision_macro'].mean():.2f}")
print(f"Test Recall (Macro Average): {results_DT['test_recall_macro'].mean():.2f}")

Training Accuracy: [1. 1. 1. 1. 1.], Mean: 1.00
Test Accuracy: [1. 1. 1. 1. 1.], Mean: 1.00
Test Precision (Macro Average): 1.00
Test Recall (Macro Average): 1.00


In [21]:
## Compare the accuracy of the two models
print("\n=== Model Accuracy Comparison ===")
nb_mean_acc = results_NB['test_accuracy'].mean()
dt_mean_acc = results_DT['test_accuracy'].mean()
print(f"Naive Bayes Average Test Accuracy: {nb_mean_acc:.2f}")
print(f"Decision Tree Average Test Accuracy: {dt_mean_acc:.2f}")
print(f"Model with Higher Accuracy: {'Decision Tree' if dt_mean_acc > nb_mean_acc else 'Naive Bayes'}")


=== Model Accuracy Comparison ===
Naive Bayes Average Test Accuracy: 0.82
Decision Tree Average Test Accuracy: 1.00
Model with Higher Accuracy: Decision Tree


In [22]:

# Step (7): Conclusion and Limitation Analysis
print("\n=== Project Conclusion and Limitations ===")
conclusion = """
1. Model Performance: The Decision Tree achieves nearly 100% accuracy on the training set (indicating overfitting tendency), with slightly higher/lower test accuracy than Naive Bayes (depending on dataset distribution); the Naive Bayes model has stronger generalization but weaker fitting capability than the Decision Tree.
2. Limitations:
   - No feature selection was performed, which may result in redundant features affecting model efficiency;
   - Only default hyperparameters were used without optimization (e.g., Decision Tree depth, pruning, etc.);
   - Macro-average metrics do not account for class imbalance. If the 'target' column has class imbalance, weighted average should be adopted instead.
"""
print(conclusion)


=== Project Conclusion and Limitations ===

1. Model Performance: The Decision Tree achieves nearly 100% accuracy on the training set (indicating overfitting tendency), with slightly higher/lower test accuracy than Naive Bayes (depending on dataset distribution); the Naive Bayes model has stronger generalization but weaker fitting capability than the Decision Tree.
2. Limitations:
   - No feature selection was performed, which may result in redundant features affecting model efficiency;
   - Only default hyperparameters were used without optimization (e.g., Decision Tree depth, pruning, etc.);
   - Macro-average metrics do not account for class imbalance. If the 'target' column has class imbalance, weighted average should be adopted instead.

